In [ ]:
# 05-latest (2024-12-01) - Profit regression model
# Step 1: Install dependencies FIRST
%pip install typing_extensions>=4.5.0 mlflow --upgrade -q

In [ ]:
# Step 2: Restart Python to pick up installed packages
# NOTE: After this cell, all variables are cleared. Run cells below manually or use "Run All Below"
dbutils.library.restartPython()

In [ ]:
# SP-5: Enhanced Profit Prediction Model
# Using sklearn on sampled data - SparkML has 100MB model limit on Serverless
#
# ENHANCEMENTS IMPLEMENTED:
# - K-Fold Cross-Validation (KFold)
# - Hyperparameter Tuning (RandomizedSearchCV)
# - Learning Curves (for bias/variance diagnosis)
# - Comprehensive Residual Analysis
# - RidgeCV/ElasticNetCV for automatic alpha tuning

import time
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, 
    RandomizedSearchCV, learning_curve
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, 
    explained_variance_score, max_error, mean_absolute_percentage_error
)
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import matplotlib.pyplot as plt
import seaborn as sns

# Set up MLflow experiment
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/leo.lwakabamba@gmail.com/poker-ml-profit-model")

print("=" * 80)
print("SP-5: Enhanced Profit Prediction Model (sklearn)")
print("=" * 80)
print("NOTE: SparkML on Serverless has 100MB limit on entire ML operation.")
print("      Using sklearn with strategic sampling instead.")
print(f"[MLflow] Experiment set")
start_time = time.time()

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

In [ ]:
# Configuration
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Model Registry (Unity Catalog three-level namespace)
MODEL_REGISTRY_PREFIX = "pokerml.default"  # catalog.schema

# Input: SP-4 output
SP4_INPUT_PATH = UC_VOLUME_DIR + 'processed/sp4_features_complete'

# Output paths
MODELS_DIR = UC_VOLUME_DIR + 'models/'
RESULTS_DIR = UC_VOLUME_DIR + 'results/'

# Training config
STREETS = ['preflop', 'flop', 'turn', 'river']
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ============================================================================
# DEBUG MODE - Read from pipeline config (UC Volume - persists across Python restarts)
# ============================================================================
import json

# UC Volume path (persists across restarts, unlike /tmp)
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

try:
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    DEBUG_MODE = config.get('debug_mode', True)
    MAX_ROWS = config.get('max_rows', 10000)
    # NEW: Use training_sample_size for model training notebooks (SP-03 and SP-05)
    TRAINING_SAMPLE_SIZE = config.get('training_sample_size', 300000)
    print(f"   Loaded config from {CONFIG_PATH}")
    print(f"   DEBUG_MODE={DEBUG_MODE}, MAX_ROWS={MAX_ROWS:,}, TRAINING_SAMPLE_SIZE={TRAINING_SAMPLE_SIZE:,}")
except FileNotFoundError:
    DEBUG_MODE = True
    MAX_ROWS = 10000
    TRAINING_SAMPLE_SIZE = 300000
    print(f"   Config not found at {CONFIG_PATH}, using defaults")

DEBUG_MAX_SAMPLES_PER_STREET = 5000  # Use 5k samples per street in debug mode
# ============================================================================

# Sample size per street for sklearn (fits in memory)
# In debug mode: use DEBUG_MAX_SAMPLES_PER_STREET
# In full mode: use TRAINING_SAMPLE_SIZE (300k) NOT MAX_ROWS (7.5M)
# This allows other notebooks (SP-06-09) to use full data while SP-03/SP-05 train on smaller sample
MAX_SAMPLES_PER_STREET = DEBUG_MAX_SAMPLES_PER_STREET if DEBUG_MODE else TRAINING_SAMPLE_SIZE

print(f"Input: {SP4_INPUT_PATH}")
print(f"Models: {MODELS_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Model Registry: {MODEL_REGISTRY_PREFIX}")
print(f"Max samples per street: {MAX_SAMPLES_PER_STREET:,}")
if DEBUG_MODE:
    print(f"\n*** DEBUG MODE ENABLED ***")
else:
    print(f"\n*** FULL MODE - using TRAINING_SAMPLE_SIZE={TRAINING_SAMPLE_SIZE:,} (not MAX_ROWS) ***")

In [ ]:
# Load SP-4 data and identify columns
print("\n[1/6] Loading SP-4 complete features...")

# Hardcoded path directly in spark.read to avoid any variable issues
print(f"   Loading from: /Volumes/pokerml/default/data/processed/sp4_features_complete")
spark_df = spark.read.parquet('/Volumes/pokerml/default/data/processed/sp4_features_complete')
total_rows = spark_df.count()
print(f"   Total rows: {total_rows:,}")
print(f"   Columns: {len(spark_df.columns)}")

# DEBUG: Print ALL columns to see what's there
print(f"\n   ALL columns in parquet file:")
for col in sorted(spark_df.columns):
    print(f"      - {col}")

# Show street distribution
print("\n   Street distribution:")
spark_df.groupBy('street').count().orderBy('count', ascending=False).show()

# Check for target column
target_col = None
if 'target_profit_bb' in spark_df.columns:
    target_col = 'target_profit_bb'
    print(f"\n   ✓ Found target column: {target_col}")
elif 'profit_bb' in spark_df.columns:
    target_col = 'profit_bb'
    print(f"\n   ✓ Found target column: {target_col}")
else:
    print("\n   ERROR: No profit column found!")
    print(f"   Looking for: 'target_profit_bb' or 'profit_bb'")
    print(f"   Has 'target_profit_bb': {'target_profit_bb' in spark_df.columns}")
    print(f"   Has 'profit_bb': {'profit_bb' in spark_df.columns}")
    raise ValueError("Missing profit column - re-run notebook 04 with profit calculation")

# ============================================================================
# TRACE PATTERN: Select a hand_id for pipeline validation
# ============================================================================
TRACE_HAND_ID = spark_df.select('hand_id').first()['hand_id']
print(f"\n[TRACE] Selected TRACE_HAND_ID: {TRACE_HAND_ID}")

# TRACE: RAW INPUT from SP-4
print(f"\n[TRACE] RAW INPUT from SP-4 for {TRACE_HAND_ID}:")
trace_cols = ['hand_id', 'actor', 'street', 'action_type', target_col]
if 'position_from_button' in spark_df.columns:
    trace_cols.append('position_from_button')
if 'position_name' in spark_df.columns:
    trace_cols.append('position_name')
spark_df.filter(F.col('hand_id') == TRACE_HAND_ID).select(trace_cols).show(20, truncate=False)

In [ ]:
# Identify feature columns
print("\n[2/6] Identifying feature columns...")

# Columns to exclude from features - avoid outcome leakage!
exclude_cols = [
    # Identity columns
    'hand_id', 'idx', 'actor', 'street', 'label',
    
    # TARGET VARIABLE - what we're predicting
    'target_profit_bb',
    
    # OUTCOME LEAKAGE - these reveal the result of the hand
    'profit_chips',          # Profit in chips (outcome)
    'profit_bb',             # Profit in BB (outcome)  
    'is_winner',             # Whether player won (outcome)
    
    # Raw card data (string columns)
    'flop', 'turn', 'river', 'hole_cards',
    
    # Categorical columns (already one-hot encoded or string)
    'predicted_bucket', 'position_bucket', 'position_name',
    
    # Action columns - these reveal what action was taken
    'action_type', 'actual_action_category', 'best_action',
    'amount',  # Bet amount reveals action
    
    # Other potential leakage
    'payout', 'total_contribution', 'final_pot', 'num_winners',
]

# Get numeric feature columns from schema
numeric_cols = []
for field in spark_df.schema.fields:
    if field.name not in exclude_cols:
        dtype_str = str(field.dataType)
        if dtype_str in ['DoubleType()', 'IntegerType()', 'LongType()', 'FloatType()']:
            numeric_cols.append(field.name)

print(f"   Numeric features: {len(numeric_cols)}")
print(f"   Sample features: {numeric_cols[:10]}")

# Verify no leakage columns made it through
leakage_check = ['profit_chips', 'is_winner', 'target_profit_bb', 'profit_bb', 'payout']
leaked = [c for c in leakage_check if c in numeric_cols]
if leaked:
    print(f"\n   *** WARNING: Potential leakage columns in features: {leaked} ***")
else:
    print(f"\n   ✓ No leakage columns detected in features")

In [ ]:
# Sample and convert to pandas per street
print("\n[3/6] Sampling data per street...")

street_data = {}
cols_to_select = numeric_cols + [target_col, 'street', 'hand_id']
available_cols = [c for c in cols_to_select if c in spark_df.columns]

for street in STREETS:
    street_spark = spark_df.filter(F.col('street') == street)
    street_count = street_spark.count()
    
    if street_count > MAX_SAMPLES_PER_STREET:
        sample_frac = MAX_SAMPLES_PER_STREET / street_count
        street_spark_sampled = street_spark.sample(fraction=sample_frac, seed=RANDOM_STATE)
        
        # DEBUG MODE: Force TRACE_HAND_ID into sample
        if DEBUG_MODE:
            trace_hand_df = street_spark.filter(F.col('hand_id') == TRACE_HAND_ID)
            if trace_hand_df.count() > 0:
                street_spark_sampled = street_spark_sampled.union(trace_hand_df).dropDuplicates(['hand_id', 'actor', 'street'])
        
        street_spark = street_spark_sampled
    
    street_pandas = street_spark.select(available_cols).toPandas()
    street_data[street] = street_pandas
    print(f"   {street}: {len(street_pandas):,} samples (of {street_count:,})")

print(f"\n   Total samples loaded: {sum(len(df) for df in street_data.values()):,}")

# TRACE: Check if TRACE_HAND_ID is in the sampled data
print(f"\n[TRACE] Checking for TRACE_HAND_ID {TRACE_HAND_ID} in sampled data:")
for street in STREETS:
    df = street_data[street]
    trace_rows = df[df['hand_id'] == TRACE_HAND_ID]
    if len(trace_rows) > 0:
        print(f"   {street}: {len(trace_rows)} rows found")
        print(trace_rows[['hand_id', 'street', target_col]].head().to_string())
    else:
        print(f"   {street}: Not present in sample")

In [ ]:
# sklearn Pipeline training function with hyperparameter tuning, learning curves, and comprehensive metrics
# UPDATED: Uses sklearn Pipeline to bundle scaler + model together
# UPDATED: Added K-Fold Cross-Validation for more reliable performance estimates
# UPDATED: Added RandomizedSearchCV for hyperparameter tuning
# UPDATED: Added learning_curve for bias/variance diagnosis
# UPDATED: Added RidgeCV/ElasticNetCV for automatic regularization tuning
# UPDATED: Each hyperparameter trial is logged as separate MLflow run with model
# UPDATED: Added GradientBoosting with conservative hyperparameters
print("\n[4/6] Setting up sklearn regressor Pipelines (scaler + model bundled)...")

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor

# ============================================================================
# TRAINING SPEED SETTINGS - CONSERVATIVE
# ============================================================================
# CONSERVATIVE (good balance of speed and thoroughness):
N_SPLITS = 3                    # 3-fold cross-validation
CV_RANDOM_STATE = 42
HYPERPARAM_CV_SPLITS = 2        # 2-fold for hyperparameter tuning
N_ITER_RANDOM_SEARCH = 8        # 8 random combinations (more thorough)
LEARNING_CURVE_POINTS = 5       # 5 points on learning curve
COMPUTE_RF_LEARNING_CURVE = True   # Compute learning curve for RF
COMPUTE_GB_LEARNING_CURVE = False  # Skip learning curve for GB (slow)
# ============================================================================

def calculate_regression_metrics(y_true, y_pred):
    """Calculate all regression metrics."""
    metrics = {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mse': mean_squared_error(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'explained_variance': explained_variance_score(y_true, y_pred),
        'max_error': max_error(y_true, y_pred),
    }
    # MAPE only if no zeros in y_true
    try:
        if (y_true != 0).all():
            metrics['mape'] = mean_absolute_percentage_error(y_true, y_pred)
        else:
            metrics['mape'] = np.nan
    except:
        metrics['mape'] = np.nan
    return metrics

def run_cross_validation(pipeline, X, y, cv, model_type):
    """Run K-Fold cross-validation and return mean ± std metrics."""
    # Define scoring metrics
    scoring = {
        'neg_rmse': 'neg_root_mean_squared_error',
        'neg_mae': 'neg_mean_absolute_error',
        'r2': 'r2',
    }
    
    print(f"      Running {N_SPLITS}-fold cross-validation...")
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, 
        return_train_score=True, n_jobs=-1
    )
    
    cv_metrics = {}
    # Convert negative metrics back to positive
    cv_metrics['cv_rmse_mean'] = -cv_results['test_neg_rmse'].mean()
    cv_metrics['cv_rmse_std'] = cv_results['test_neg_rmse'].std()
    cv_metrics['cv_mae_mean'] = -cv_results['test_neg_mae'].mean()
    cv_metrics['cv_mae_std'] = cv_results['test_neg_mae'].std()
    cv_metrics['cv_r2_mean'] = cv_results['test_r2'].mean()
    cv_metrics['cv_r2_std'] = cv_results['test_r2'].std()
    
    # Training metrics
    cv_metrics['cv_train_rmse_mean'] = -cv_results['train_neg_rmse'].mean()
    cv_metrics['cv_train_r2_mean'] = cv_results['train_r2'].mean()
    
    print(f"      CV RMSE: {cv_metrics['cv_rmse_mean']:.4f} ± {cv_metrics['cv_rmse_std']:.4f}")
    print(f"      CV MAE:  {cv_metrics['cv_mae_mean']:.4f} ± {cv_metrics['cv_mae_std']:.4f}")
    print(f"      CV R²:   {cv_metrics['cv_r2_mean']:.4f} ± {cv_metrics['cv_r2_std']:.4f}")
    
    # Check for overfitting (train >> test)
    train_test_gap = cv_metrics['cv_train_r2_mean'] - cv_metrics['cv_r2_mean']
    if train_test_gap > 0.1:
        print(f"      ⚠️ Potential overfitting: Train-Test R² gap = {train_test_gap:.4f}")
    
    return cv_metrics

def format_params_for_name(params):
    """Format parameters into a short readable string for run names."""
    parts = []
    for k, v in params.items():
        # Extract just the parameter name (after __)
        short_key = k.split('__')[-1]
        # Format value
        if isinstance(v, float):
            parts.append(f"{short_key}={v:.2g}")
        else:
            parts.append(f"{short_key}={v}")
    return "_".join(parts)

def run_hyperparameter_tuning_with_logging(base_pipeline, param_grid, X, y, cv, model_type, street, feature_names):
    """
    Run hyperparameter tuning and log EACH trial as a separate MLflow run.
    Each run includes the model, parameters, and performance metrics.
    """
    from sklearn.model_selection import ParameterSampler
    
    print(f"      Running hyperparameter tuning ({N_ITER_RANDOM_SEARCH} iterations)...")
    print(f"      Each trial will be logged as a separate MLflow run with model...")
    
    # Sample parameters
    param_list = list(ParameterSampler(param_grid, n_iter=N_ITER_RANDOM_SEARCH, random_state=CV_RANDOM_STATE))
    
    # Create sample input for signature
    sample_input = pd.DataFrame(X[:5].values if hasattr(X, 'values') else X[:5], columns=feature_names)
    
    trial_results = []
    best_score = np.inf  # For RMSE, lower is better
    best_pipeline = None
    best_params = None
    
    for i, params in enumerate(param_list):
        # Clone base pipeline and set parameters
        pipeline = clone(base_pipeline)
        pipeline.set_params(**params)
        
        # Format run name with parameters
        param_str = format_params_for_name(params)
        run_name = f"{street}_{model_type}_trial{i+1}_{param_str}"
        
        # Cross-validate this configuration
        scoring = {'neg_rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}
        cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
        
        mean_rmse = -cv_results['test_neg_rmse'].mean()
        std_rmse = cv_results['test_neg_rmse'].std()
        mean_r2 = cv_results['test_r2'].mean()
        
        print(f"         Trial {i+1}/{N_ITER_RANDOM_SEARCH}: RMSE={mean_rmse:.4f}±{std_rmse:.4f} | {param_str}")
        
        # Log this trial as a separate MLflow run
        with mlflow.start_run(run_name=run_name, nested=True):
            # Log parameters
            mlflow.log_param("street", street)
            mlflow.log_param("model_type", model_type)
            mlflow.log_param("trial_number", i + 1)
            mlflow.log_param("is_best", False)  # Will update for best
            
            for k, v in params.items():
                mlflow.log_param(k, str(v))
            
            # Log CV metrics
            mlflow.log_metric("cv_rmse_mean", mean_rmse)
            mlflow.log_metric("cv_rmse_std", std_rmse)
            mlflow.log_metric("cv_r2_mean", mean_r2)
            mlflow.log_metric("cv_train_rmse_mean", -cv_results['train_neg_rmse'].mean())
            
            # Fit on full training data and log model
            pipeline.fit(X, y)
            
            # Create signature and log model
            signature = infer_signature(sample_input, pipeline.predict(X[:5]))
            mlflow.sklearn.log_model(pipeline, artifact_path="model", signature=signature)
        
        # Track results
        trial_results.append({
            'params': params,
            'mean_rmse': mean_rmse,
            'std_rmse': std_rmse,
            'mean_r2': mean_r2,
            'pipeline': pipeline
        })
        
        # Track best (lower RMSE is better)
        if mean_rmse < best_score:
            best_score = mean_rmse
            best_pipeline = pipeline
            best_params = params
    
    # Log the best trial with is_best=True flag
    param_str = format_params_for_name(best_params)
    with mlflow.start_run(run_name=f"{street}_{model_type}_BEST_{param_str}", nested=True):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("is_best", True)
        for k, v in best_params.items():
            mlflow.log_param(k, str(v))
        mlflow.log_metric("cv_rmse_mean", best_score)
        
        signature = infer_signature(sample_input, best_pipeline.predict(X[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
    
    print(f"      Best params: {best_params}")
    print(f"      Best CV RMSE: {best_score:.4f}")
    
    # Prepare tuning results for artifact logging
    tuning_results = {
        'best_score': best_score,
        'best_params': {k: str(v) for k, v in best_params.items()},
        'all_trials': [
            {'params': {k: str(v) for k, v in t['params'].items()}, 
             'cv_rmse_mean': t['mean_rmse'],
             'cv_rmse_std': t['std_rmse'],
             'cv_r2_mean': t['mean_r2']}
            for t in trial_results
        ]
    }
    
    return best_pipeline, best_params, tuning_results

def plot_learning_curve(estimator, X, y, cv, street, model_type, train_sizes=None):
    """
    Plot learning curve to diagnose bias/variance.
    
    - Both curves high and close together = underfitting (high bias)
    - Training low, validation high = overfitting (high variance)
    - Both curves converging at good score = well-fit model
    """
    if train_sizes is None:
        train_sizes = np.linspace(0.1, 1.0, LEARNING_CURVE_POINTS)
    
    print(f"      Computing learning curve (this may take a moment)...")
    
    try:
        train_sizes_abs, train_scores, val_scores = learning_curve(
            estimator, X, y, cv=cv, n_jobs=-1,
            train_sizes=train_sizes,
            scoring='neg_root_mean_squared_error',
            random_state=CV_RANDOM_STATE
        )
        
        # Convert negative scores to positive RMSE
        train_mean = -train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        val_mean = -val_scores.mean(axis=1)
        val_std = val_scores.std(axis=1)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot training scores
        ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
        ax.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Training RMSE')
        
        # Plot validation scores
        ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
        ax.plot(train_sizes_abs, val_mean, 'o-', color='orange', label='Cross-validation RMSE')
        
        ax.set_xlabel('Training Set Size', fontsize=12)
        ax.set_ylabel('RMSE (lower is better)', fontsize=12)
        ax.set_title(f'Learning Curve - {street.upper()} ({model_type})', fontsize=14)
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Add interpretation annotation
        final_gap = val_mean[-1] - train_mean[-1]
        if final_gap > 0.5:  # Arbitrary threshold for regression
            ax.annotate(f'Gap: {final_gap:.3f} (possible overfitting)', 
                       xy=(train_sizes_abs[-1], val_mean[-1]),
                       xytext=(train_sizes_abs[-1] * 0.7, val_mean[-1] + 0.5),
                       arrowprops=dict(arrowstyle='->', color='red'),
                       fontsize=10, color='red')
        
        plt.tight_layout()
        
        # Return metrics for logging
        learning_metrics = {
            'lc_final_train_rmse': float(train_mean[-1]),
            'lc_final_val_rmse': float(val_mean[-1]),
            'lc_train_val_gap': float(final_gap),
            'lc_train_std': float(train_std[-1]),
            'lc_val_std': float(val_std[-1]),
        }
        
        return fig, learning_metrics
    
    except Exception as e:
        print(f"      Warning: Could not compute learning curve: {e}")
        return None, {}

def plot_regression_diagnostics(y_true, y_pred, street, model_type):
    """Plot actual vs predicted and residual plots."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Actual vs Predicted
    ax1 = axes[0]
    ax1.scatter(y_true, y_pred, alpha=0.3, s=10)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax1.set_xlabel('Actual Profit (BB)', fontsize=12)
    ax1.set_ylabel('Predicted Profit (BB)', fontsize=12)
    ax1.set_title(f'Actual vs Predicted - {street.upper()} ({model_type})', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Residual plot
    ax2 = axes[1]
    residuals = y_true - y_pred
    ax2.scatter(y_pred, residuals, alpha=0.3, s=10)
    ax2.axhline(y=0, color='r', linestyle='--', lw=2)
    ax2.set_xlabel('Predicted Profit (BB)', fontsize=12)
    ax2.set_ylabel('Residual (BB)', fontsize=12)
    ax2.set_title(f'Residual Plot - {street.upper()} ({model_type})', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_residual_analysis(y_true, y_pred, street, model_type):
    """Plot comprehensive residual analysis including distribution and Q-Q plot."""
    residuals = y_true - y_pred
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Residual distribution (histogram)
    ax1 = axes[0, 0]
    ax1.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    ax1.axvline(x=0, color='r', linestyle='--', lw=2, label='Zero')
    ax1.axvline(x=residuals.mean(), color='g', linestyle='-', lw=2, label=f'Mean: {residuals.mean():.2f}')
    ax1.set_xlabel('Residual (BB)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.set_title(f'Residual Distribution - {street.upper()} ({model_type})', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Residuals vs Predicted
    ax2 = axes[0, 1]
    ax2.scatter(y_pred, residuals, alpha=0.3, s=10)
    ax2.axhline(y=0, color='r', linestyle='--', lw=2)
    # Add smoothed trend line
    try:
        from scipy.stats import binned_statistic
        bins = 20
        bin_means, bin_edges, _ = binned_statistic(y_pred, residuals, statistic='mean', bins=bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        ax2.plot(bin_centers, bin_means, 'g-', lw=2, label='Trend')
    except:
        pass
    ax2.set_xlabel('Predicted Profit (BB)', fontsize=12)
    ax2.set_ylabel('Residual (BB)', fontsize=12)
    ax2.set_title(f'Residuals vs Predicted - {street.upper()} ({model_type})', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    # 3. Q-Q plot for normality check
    ax3 = axes[1, 0]
    stats.probplot(residuals, dist="norm", plot=ax3)
    ax3.set_title(f'Q-Q Plot (Normality Check) - {street.upper()} ({model_type})', fontsize=14)
    ax3.grid(True, alpha=0.3)
    
    # 4. Scale-Location plot (sqrt of abs residuals vs predicted)
    ax4 = axes[1, 1]
    sqrt_abs_residuals = np.sqrt(np.abs(residuals))
    ax4.scatter(y_pred, sqrt_abs_residuals, alpha=0.3, s=10)
    # Add smoothed trend line
    try:
        from scipy.stats import binned_statistic
        bin_means, bin_edges, _ = binned_statistic(y_pred, sqrt_abs_residuals, statistic='mean', bins=bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        ax4.plot(bin_centers, bin_means, 'r-', lw=2, label='Trend')
    except:
        pass
    ax4.set_xlabel('Predicted Profit (BB)', fontsize=12)
    ax4.set_ylabel('√|Residual|', fontsize=12)
    ax4.set_title(f'Scale-Location Plot (Heteroscedasticity) - {street.upper()} ({model_type})', fontsize=14)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_feature_importance(regressor, feature_names, street, model_type):
    """Plot and return feature importance chart."""
    importances = None
    
    if hasattr(regressor, 'feature_importances_'):
        importances = regressor.feature_importances_
    elif hasattr(regressor, 'coef_'):
        importances = np.abs(regressor.coef_)
    
    if importances is None:
        return None, None
    
    # Validate lengths match
    if len(importances) != len(feature_names):
        min_len = min(len(feature_names), len(importances))
        feature_names = feature_names[:min_len]
        importances = importances[:min_len]
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    top_n = min(20, len(importance_df))
    top_features = importance_df.head(top_n)
    
    y_pos = np.arange(top_n)
    ax.barh(y_pos, top_features['importance'].values, align='center', color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features['feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'Feature Importance - {street.upper()} ({model_type})', fontsize=14)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    return fig, importance_df

def train_sklearn_pipelines(X_train, X_test, y_train, y_test, street, feature_cols):
    """
    Train multiple sklearn Pipelines (scaler + model bundled together).
    
    Returns Pipeline objects that accept RAW features - no manual scaling needed at inference.
    
    FEATURES:
    - K-Fold Cross-Validation for reliable performance estimates
    - RandomizedSearchCV for hyperparameter tuning (RF, GB)
    - RidgeCV/ElasticNetCV for automatic regularization tuning
    - Learning curves for bias/variance diagnosis
    - Each hyperparameter trial logged as separate MLflow run with model
    - Feature importance logged to each model run
    """
    
    results = []
    learning_curve_figures = {}
    
    # Create sample input for signature (RAW features, not scaled)
    sample_input = pd.DataFrame(X_train[:5].values, columns=feature_cols)
    
    # Set up K-Fold CV (not stratified for regression)
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)
    cv_tune = KFold(n_splits=HYPERPARAM_CV_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)
    
    # =========================================================================
    # Model 1: Ridge Regression with RidgeCV for automatic alpha tuning
    # =========================================================================
    print("\n   Training Ridge Pipeline with automatic alpha tuning (RidgeCV)...")
    with mlflow.start_run(run_name=f"{street}_Ridge_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "Ridge")
        mlflow.log_param("pipeline", "StandardScaler + Ridge")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("regularization", "L2")
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("alpha_tuning", "RidgeCV")
        
        # Use RidgeCV to automatically find best alpha
        alphas = np.logspace(-3, 3, 50)  # 0.001 to 1000
        
        # First, find best alpha using RidgeCV (on scaled data)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        ridge_cv = RidgeCV(alphas=alphas, cv=5)
        ridge_cv.fit(X_train_scaled, y_train)
        best_alpha = ridge_cv.alpha_
        
        print(f"      Best alpha found: {best_alpha:.4f}")
        mlflow.log_param("best_alpha", best_alpha)
        
        # Create pipeline with best alpha
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', Ridge(alpha=best_alpha, random_state=RANDOM_STATE))
        ])
        
        # Run K-Fold cross-validation
        cv_metrics = run_cross_validation(pipeline, X_train, y_train, cv, 'Ridge')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        # Compute learning curve (always for Ridge - it's fast)
        lc_fig, lc_metrics = plot_learning_curve(
            pipeline, X_train, y_train, cv, street, 'Ridge'
        )
        if lc_fig:
            mlflow.log_figure(lc_fig, f"learning_curve_{street}_ridge.png")
            for name, value in lc_metrics.items():
                mlflow.log_metric(name, value)
            learning_curve_figures[f'Ridge_{street}'] = lc_fig
            plt.close(lc_fig)
        
        # Fit on full training set and evaluate on test set
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        metrics = calculate_regression_metrics(y_test.values, y_pred)
        for name, value in metrics.items():
            if not np.isnan(value):
                mlflow.log_metric(f"test_{name}", value)
        
        # Log feature importance
        try:
            regressor = pipeline.named_steps['regressor']
            fi_fig, fi_df = plot_feature_importance(regressor, feature_cols, street, 'Ridge')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_ridge.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_ridge.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        # Signature uses RAW input (pipeline handles scaling internally)
        signature = infer_signature(sample_input, pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics, 'best_alpha': best_alpha}
        results.append({
            'pipeline': pipeline, 'model_type': 'Ridge', 'y_pred': y_pred, 
            'best_params': {'alpha': best_alpha}, **all_metrics
        })
        print(f"      Test: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
    
    # =========================================================================
    # Model 2: ElasticNet with ElasticNetCV for automatic tuning
    # =========================================================================
    print("\n   Training ElasticNet Pipeline with automatic tuning (ElasticNetCV)...")
    with mlflow.start_run(run_name=f"{street}_ElasticNet_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "ElasticNet")
        mlflow.log_param("pipeline", "StandardScaler + ElasticNet")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("regularization", "L1+L2")
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("alpha_tuning", "ElasticNetCV")
        
        # Use ElasticNetCV to find best alpha and l1_ratio
        alphas = np.logspace(-3, 2, 30)  # 0.001 to 100
        l1_ratios = [0.1, 0.5, 0.9]  # Reduced from 5 to 3 values
        
        enet_cv = ElasticNetCV(
            alphas=alphas,
            l1_ratio=l1_ratios,
            cv=5,
            random_state=RANDOM_STATE,
            max_iter=5000
        )
        enet_cv.fit(X_train_scaled, y_train)  # Use pre-scaled data
        best_alpha = enet_cv.alpha_
        best_l1_ratio = enet_cv.l1_ratio_
        
        print(f"      Best alpha: {best_alpha:.4f}, Best l1_ratio: {best_l1_ratio:.2f}")
        mlflow.log_param("best_alpha", best_alpha)
        mlflow.log_param("best_l1_ratio", best_l1_ratio)
        
        # Create pipeline with best parameters
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', ElasticNet(
                alpha=best_alpha, 
                l1_ratio=best_l1_ratio, 
                random_state=RANDOM_STATE, 
                max_iter=5000
            ))
        ])
        
        # Run K-Fold cross-validation
        cv_metrics = run_cross_validation(pipeline, X_train, y_train, cv, 'ElasticNet')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        # Compute learning curve (always for ElasticNet - it's fast)
        lc_fig, lc_metrics = plot_learning_curve(
            pipeline, X_train, y_train, cv, street, 'ElasticNet'
        )
        if lc_fig:
            mlflow.log_figure(lc_fig, f"learning_curve_{street}_elasticnet.png")
            for name, value in lc_metrics.items():
                mlflow.log_metric(name, value)
            learning_curve_figures[f'ElasticNet_{street}'] = lc_fig
            plt.close(lc_fig)
        
        # Fit on full training set and evaluate on test set
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        metrics = calculate_regression_metrics(y_test.values, y_pred)
        for name, value in metrics.items():
            if not np.isnan(value):
                mlflow.log_metric(f"test_{name}", value)
        
        # Log feature importance
        try:
            regressor = pipeline.named_steps['regressor']
            fi_fig, fi_df = plot_feature_importance(regressor, feature_cols, street, 'ElasticNet')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_elasticnet.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_elasticnet.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics, 'best_alpha': best_alpha, 'best_l1_ratio': best_l1_ratio}
        results.append({
            'pipeline': pipeline, 'model_type': 'ElasticNet', 'y_pred': y_pred,
            'best_params': {'alpha': best_alpha, 'l1_ratio': best_l1_ratio}, **all_metrics
        })
        print(f"      Test: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
    
    # =========================================================================
    # Model 3: Random Forest with hyperparameter tuning (each trial logged)
    # =========================================================================
    print("\n   Training RandomForest Pipeline with hyperparameter tuning...")
    with mlflow.start_run(run_name=f"{street}_RandomForest_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("pipeline", "StandardScaler + RandomForest")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("hyperparam_tuning", True)
        mlflow.log_param("n_iter_search", N_ITER_RANDOM_SEARCH)
        
        # Define base pipeline
        base_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
        
        # Define hyperparameter grid
        param_grid = {
            'regressor__n_estimators': [100, 150, 200],
            'regressor__max_depth': [8, 12, 15, None],
            'regressor__min_samples_split': [2, 5, 10],
            'regressor__min_samples_leaf': [1, 2, 4],
            'regressor__max_features': ['sqrt', 0.5],
        }
        
        # Run hyperparameter tuning with individual run logging
        best_pipeline, best_params, tuning_results = run_hyperparameter_tuning_with_logging(
            base_pipeline, param_grid, X_train, y_train, cv_tune, 'RF', street, feature_cols
        )
        
        # Log tuning results
        mlflow.log_params({f"best_{k}": str(v) for k, v in best_params.items()})
        mlflow.log_metric("tuning_best_rmse", tuning_results['best_score'])
        mlflow.log_dict(tuning_results, "hyperparameter_tuning_results.json")
        
        # Run K-Fold cross-validation on best model
        cv_metrics = run_cross_validation(best_pipeline, X_train, y_train, cv, 'RF')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        # Compute learning curve (conditionally - RF is slow)
        lc_metrics = {}
        if COMPUTE_RF_LEARNING_CURVE:
            lc_fig, lc_metrics = plot_learning_curve(
                best_pipeline, X_train, y_train, cv, street, 'RF'
            )
            if lc_fig:
                mlflow.log_figure(lc_fig, f"learning_curve_{street}_rf.png")
                for name, value in lc_metrics.items():
                    mlflow.log_metric(name, value)
                learning_curve_figures[f'RF_{street}'] = lc_fig
                plt.close(lc_fig)
        else:
            print("      Skipping RF learning curve (COMPUTE_RF_LEARNING_CURVE=False)")
        
        # Fit on full training set and evaluate on test set
        best_pipeline.fit(X_train, y_train)
        y_pred = best_pipeline.predict(X_test)
        
        metrics = calculate_regression_metrics(y_test.values, y_pred)
        for name, value in metrics.items():
            if not np.isnan(value):
                mlflow.log_metric(f"test_{name}", value)
        
        # Log feature importance
        try:
            regressor = best_pipeline.named_steps['regressor']
            fi_fig, fi_df = plot_feature_importance(regressor, feature_cols, street, 'RF')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_rf.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_rf.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, best_pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics}
        results.append({
            'pipeline': best_pipeline, 'model_type': 'RandomForest', 'y_pred': y_pred,
            'best_params': best_params, **all_metrics
        })
        print(f"      Test: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
    
    # =========================================================================
    # Model 4: Gradient Boosting with hyperparameter tuning (NEW)
    # =========================================================================
    print("\n   Training GradientBoosting Pipeline with hyperparameter tuning...")
    with mlflow.start_run(run_name=f"{street}_GradientBoosting_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "GradientBoosting")
        mlflow.log_param("pipeline", "StandardScaler + GradientBoosting")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("hyperparam_tuning", True)
        mlflow.log_param("n_iter_search", N_ITER_RANDOM_SEARCH)
        
        # Define base pipeline with early stopping
        base_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', GradientBoostingRegressor(
                random_state=RANDOM_STATE,
                validation_fraction=0.1,
                n_iter_no_change=10,  # Early stopping
                tol=1e-4
            ))
        ])
        
        # Conservative hyperparameter grid
        param_grid = {
            'regressor__n_estimators': [100, 150, 200],
            'regressor__max_depth': [3, 5, 7],
            'regressor__learning_rate': [0.05, 0.1, 0.15],
            'regressor__subsample': [0.8, 1.0],
            'regressor__min_samples_split': [2, 5],
            'regressor__min_samples_leaf': [1, 2],
        }
        
        # Run hyperparameter tuning with individual run logging
        best_pipeline, best_params, tuning_results = run_hyperparameter_tuning_with_logging(
            base_pipeline, param_grid, X_train, y_train, cv_tune, 'GB', street, feature_cols
        )
        
        # Log tuning results
        mlflow.log_params({f"best_{k}": str(v) for k, v in best_params.items()})
        mlflow.log_metric("tuning_best_rmse", tuning_results['best_score'])
        mlflow.log_dict(tuning_results, "hyperparameter_tuning_results.json")
        
        # Run K-Fold cross-validation on best model
        cv_metrics = run_cross_validation(best_pipeline, X_train, y_train, cv, 'GB')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        # Learning curve (conditionally - GB can be slow)
        lc_metrics = {}
        if COMPUTE_GB_LEARNING_CURVE:
            lc_fig, lc_metrics = plot_learning_curve(
                best_pipeline, X_train, y_train, cv, street, 'GB'
            )
            if lc_fig:
                mlflow.log_figure(lc_fig, f"learning_curve_{street}_gb.png")
                for name, value in lc_metrics.items():
                    mlflow.log_metric(name, value)
                learning_curve_figures[f'GB_{street}'] = lc_fig
                plt.close(lc_fig)
        else:
            print("      Skipping GB learning curve (COMPUTE_GB_LEARNING_CURVE=False)")
        
        # Fit on full training set and evaluate on test set
        best_pipeline.fit(X_train, y_train)
        y_pred = best_pipeline.predict(X_test)
        
        metrics = calculate_regression_metrics(y_test.values, y_pred)
        for name, value in metrics.items():
            if not np.isnan(value):
                mlflow.log_metric(f"test_{name}", value)
        
        # Log feature importance
        try:
            regressor = best_pipeline.named_steps['regressor']
            fi_fig, fi_df = plot_feature_importance(regressor, feature_cols, street, 'GB')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_gb.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_gb.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, best_pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics}
        results.append({
            'pipeline': best_pipeline, 'model_type': 'GradientBoosting', 'y_pred': y_pred,
            'best_params': best_params, **all_metrics
        })
        print(f"      Test: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
    
    # Return best pipeline by CV RMSE (more reliable than single test set)
    best = min(results, key=lambda x: x.get('cv_rmse_mean', x['rmse']))
    return best, results, learning_curve_figures

print("   sklearn Pipeline training function ready")
print("   *** IMPORTANT: Pipelines bundle StandardScaler + Model together ***")
print("   *** At inference, pass RAW features - no manual scaling needed! ***")
print("   ")
print(f"   *** K-FOLD CROSS-VALIDATION: {N_SPLITS}-fold KFold ***")
print(f"   *** HYPERPARAMETER TUNING: {N_ITER_RANDOM_SEARCH} trials, each logged to MLflow ***")
print("   *** REGULARIZATION TUNING: RidgeCV/ElasticNetCV for automatic alpha ***")
print(f"   *** LEARNING CURVES: {LEARNING_CURVE_POINTS} points (RF: {'enabled' if COMPUTE_RF_LEARNING_CURVE else 'disabled'}, GB: {'enabled' if COMPUTE_GB_LEARNING_CURVE else 'disabled'}) ***")
print("   *** FEATURE IMPORTANCE: Logged to each model run ***")
print("   ")
print("   Active models:")
print("   - Ridge (auto-tuned alpha via RidgeCV)")
print("   - ElasticNet (auto-tuned alpha + l1_ratio via ElasticNetCV)")
print("   - RandomForest (tuned hyperparameters via RandomizedSearchCV, each trial logged)")
print("   - GradientBoosting (tuned hyperparameters, early stopping)")

In [ ]:
# Train models per street with diagnostic plots, learning curves, and hyperparameter tuning
# UPDATED: Uses Pipeline (scaler bundled), saves scaling parameters
# UPDATED: Added comprehensive residual analysis plots
# UPDATED: Captures learning curve figures from training function
print("\n[5/6] Training street-specific regressor Pipelines...")

best_models = {}
all_results = []
diagnostic_figures = {}
residual_figures = {}
all_learning_curves = {}  # Store learning curves for display

for street in STREETS:
    print(f"\n{'=' * 80}")
    print(f"STREET: {street.upper()}")
    print(f"{'=' * 80}")
    
    df = street_data[street]
    
    if len(df) < 100:
        print(f"   Skipping - only {len(df)} samples")
        continue
    
    # Prepare features and target
    feature_cols = [c for c in numeric_cols if c in df.columns]
    
    # Get target
    y = df[target_col].copy()
    X = df[feature_cols].copy()
    
    # Clean data
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    X = X.clip(lower=-1e9, upper=1e9)
    
    # Remove rows with invalid target
    valid_mask = y.notna() & np.isfinite(y)
    X = X[valid_mask]
    y = y[valid_mask]
    
    print(f"   Samples: {len(X):,}")
    print(f"   Features: {len(feature_cols)}")
    print(f"   Target range: [{y.min():.2f}, {y.max():.2f}] BB")
    print(f"   Target mean: {y.mean():.2f}, std: {y.std():.2f}")
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"   Train: {len(X_train):,} | Test: {len(X_test):,}")
    
    # Train pipelines - pass feature names for signature
    # Now returns 3 values: best, all_results, learning_curve_figures
    best, all_model_results, learning_curve_figures = train_sklearn_pipelines(
        X_train, X_test, y_train, y_test, street, feature_cols
    )
    
    # Store learning curves
    all_learning_curves.update(learning_curve_figures)
    
    print(f"\n   [BEST] {best['model_type']} -> CV RMSE: {best.get('cv_rmse_mean', best['rmse']):.4f}, Test R2: {best['r2']:.4f}")
    if 'best_params' in best:
        print(f"   [BEST PARAMS] {best['best_params']}")
    
    # Get pipeline from best result
    pipeline = best['pipeline']
    
    # Extract scaler from pipeline for saving scaling params
    scaler = pipeline.named_steps['scaler']
    
    # Plot and display diagnostic plots for best model
    try:
        fig = plot_regression_diagnostics(y_test.values, best['y_pred'], street, best['model_type'])
        diagnostic_figures[street] = fig
        display(fig)  # Show in notebook
        plt.close(fig)
    except Exception as e:
        print(f"   Warning: Could not plot diagnostics: {e}")
    
    # Plot and display comprehensive residual analysis
    try:
        residual_fig = plot_residual_analysis(y_test.values, best['y_pred'], street, best['model_type'])
        residual_figures[street] = residual_fig
        display(residual_fig)  # Show in notebook
        plt.close(residual_fig)
    except Exception as e:
        print(f"   Warning: Could not plot residual analysis: {e}")
    
    # Store best model info (now stores pipeline instead of separate model/scaler)
    best_models[street] = {
        'pipeline': pipeline,
        'model_type': best['model_type'],
        'best_params': best.get('best_params', {}),
        'cv_rmse_mean': best.get('cv_rmse_mean', best['rmse']),
        'cv_rmse_std': best.get('cv_rmse_std', 0),
        'cv_r2_mean': best.get('cv_r2_mean', best['r2']),
        'cv_r2_std': best.get('cv_r2_std', 0),
        'lc_train_val_gap': best.get('lc_train_val_gap', 0),
        'rmse': best['rmse'],
        'mse': best['mse'],
        'mae': best['mae'],
        'r2': best['r2'],
        'explained_variance': best['explained_variance'],
        'max_error': best['max_error'],
        'features': feature_cols
    }
    
    # Track results with all metrics (CV, test, and learning curve)
    all_results.append({
        'street': street,
        'model_type': best['model_type'],
        'best_params': str(best.get('best_params', {})),
        'cv_rmse_mean': float(best.get('cv_rmse_mean', best['rmse'])),
        'cv_rmse_std': float(best.get('cv_rmse_std', 0)),
        'cv_r2_mean': float(best.get('cv_r2_mean', best['r2'])),
        'cv_r2_std': float(best.get('cv_r2_std', 0)),
        'lc_final_train_rmse': float(best.get('lc_final_train_rmse', 0)),
        'lc_final_val_rmse': float(best.get('lc_final_val_rmse', 0)),
        'lc_train_val_gap': float(best.get('lc_train_val_gap', 0)),
        'test_rmse': float(best['rmse']),
        'test_mse': float(best['mse']),
        'test_mae': float(best['mae']),
        'test_r2': float(best['r2']),
        'test_explained_variance': float(best['explained_variance']),
        'test_max_error': float(best['max_error']),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'n_features': len(feature_cols)
    })
    
    # ========================================================================
    # Register best PIPELINE in MLflow Model Registry
    # Pipeline includes scaler - webapp can pass RAW features directly
    # ========================================================================
    model_name = f"pokerml.default.02-profit-modeling-{street}-v3"
    
    with mlflow.start_run(run_name=f"{street}_best_pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", best['model_type'])
        mlflow.log_param("pipeline", "StandardScaler + " + best['model_type'])
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("features", str(feature_cols[:20]))  # Log first 20
        mlflow.log_param("cv_folds", N_SPLITS)
        
        # Log best hyperparameters
        if 'best_params' in best:
            for k, v in best['best_params'].items():
                mlflow.log_param(f"best_{k}", str(v))
        
        # Log CV metrics
        mlflow.log_metric("cv_rmse_mean", best.get('cv_rmse_mean', best['rmse']))
        mlflow.log_metric("cv_rmse_std", best.get('cv_rmse_std', 0))
        mlflow.log_metric("cv_r2_mean", best.get('cv_r2_mean', best['r2']))
        mlflow.log_metric("cv_r2_std", best.get('cv_r2_std', 0))
        
        # Log learning curve metrics
        if 'lc_train_val_gap' in best:
            mlflow.log_metric("lc_train_val_gap", best['lc_train_val_gap'])
            mlflow.log_metric("lc_final_train_rmse", best.get('lc_final_train_rmse', 0))
            mlflow.log_metric("lc_final_val_rmse", best.get('lc_final_val_rmse', 0))
        
        # Log test metrics
        mlflow.log_metric("test_rmse", best['rmse'])
        mlflow.log_metric("test_mse", best['mse'])
        mlflow.log_metric("test_mae", best['mae'])
        mlflow.log_metric("test_r2", best['r2'])
        mlflow.log_metric("test_explained_variance", best['explained_variance'])
        mlflow.log_metric("test_max_error", best['max_error'])
        if not np.isnan(best.get('mape', np.nan)):
            mlflow.log_metric("test_mape", best['mape'])
        
        # Create signature with RAW input (pipeline handles scaling)
        sample_input = pd.DataFrame(X_train[:5].values, columns=feature_cols)
        signature = infer_signature(sample_input, pipeline.predict(X_train[:5]))
        
        # Log the PIPELINE (includes scaler) to Unity Catalog registry
        mlflow.sklearn.log_model(
            pipeline,  # Pipeline, not just model
            artifact_path="model",
            signature=signature,
            registered_model_name=model_name
        )
        
        # ====================================================================
        # SAVE SCALING PARAMETERS as artifact (for debugging/backup)
        # ====================================================================
        scaling_params = {
            'feature_names': feature_cols,
            'means': scaler.mean_.tolist(),
            'stds': scaler.scale_.tolist(),
            'street': street,
            'model_type': best['model_type'],
            'best_params': best.get('best_params', {}),
            'model_version': 'v3',
            'note': 'Pipeline includes scaler - these params are for reference only'
        }
        mlflow.log_dict(scaling_params, "scaling_params.json")
        
        # Log diagnostic figures if available
        if street in diagnostic_figures:
            mlflow.log_figure(diagnostic_figures[street], f"diagnostics_{street}.png")
        if street in residual_figures:
            mlflow.log_figure(residual_figures[street], f"residual_analysis_{street}.png")
        
        print(f"   Registered PIPELINE in MLflow: {model_name}")
        print(f"   *** Pipeline includes StandardScaler - pass RAW features at inference ***")
    
    # Save metadata + scaling params to UC Volume - HARDCODED PATH
    model_metadata = {
        'model_type': best['model_type'],
        'pipeline': 'StandardScaler + ' + best['model_type'],
        'best_params': str(best.get('best_params', {})),
        'features': feature_cols[:50],  # Limit for JSON
        'scaling_means': scaler.mean_.tolist()[:50],  # Limit for JSON
        'scaling_stds': scaler.scale_.tolist()[:50],  # Limit for JSON
        'n_features': len(feature_cols),
        'cv_rmse_mean': float(best.get('cv_rmse_mean', best['rmse'])),
        'cv_rmse_std': float(best.get('cv_rmse_std', 0)),
        'cv_r2_mean': float(best.get('cv_r2_mean', best['r2'])),
        'lc_train_val_gap': float(best.get('lc_train_val_gap', 0)),
        'test_rmse': float(best['rmse']),
        'test_mse': float(best['mse']),
        'test_mae': float(best['mae']),
        'test_r2': float(best['r2']),
        'test_explained_variance': float(best['explained_variance']),
        'test_max_error': float(best['max_error']),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'note': 'Pipeline includes scaler - scaling params saved for reference'
    }
    
    metadata_df = spark.createDataFrame([model_metadata])
    metadata_df.write.mode('overwrite').json(f'/Volumes/pokerml/default/data/models/sp5_{street}_metadata')
    print(f"   Saved metadata + scaling params: /Volumes/pokerml/default/data/models/sp5_{street}_metadata")

# Save results comparison - HARDCODED PATH
if all_results:
    results_df = spark.createDataFrame(all_results)
    results_df.write.mode('overwrite').parquet('/Volumes/pokerml/default/data/results/sp5_model_comparison')
    print(f"\n   Saved results: /Volumes/pokerml/default/data/results/sp5_model_comparison")

print("\n" + "=" * 80)
print("IMPORTANT: Models are now sklearn Pipelines with scaler included!")
print("At inference, pass RAW features directly - no manual scaling needed.")
print("=" * 80)

In [ ]:
# Feature Importance Analysis
# UPDATED: Extract regressor from pipeline to get feature importances
print("\n[6/6] Feature Importance Analysis...")

feature_importance_results = {}

for street, model_info in best_models.items():
    pipeline = model_info['pipeline']
    features = model_info['features']
    
    # Extract the regressor from the pipeline
    regressor = pipeline.named_steps['regressor']
    
    print(f"\n   {street.upper()} - {model_info['model_type']} Top 20 Features:")
    
    # Get feature importances
    if hasattr(regressor, 'feature_importances_'):
        importances = regressor.feature_importances_
    elif hasattr(regressor, 'coef_'):
        importances = np.abs(regressor.coef_)
    else:
        print(f"      No feature importances available")
        continue
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': features[:len(importances)],
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    feature_importance_results[street] = importance_df
    
    # Display top 20
    for _, row in importance_df.head(20).iterrows():
        print(f"      {row['feature']:40s}: {row['importance']:.4f}")
    
    # Log to MLflow
    with mlflow.start_run(run_name=f"{street}_feature_importance"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", model_info['model_type'])
        for i, (_, row) in enumerate(importance_df.head(20).iterrows()):
            mlflow.log_metric(f"importance_{i+1}", float(row['importance']))
            mlflow.log_param(f"feature_{i+1}", row['feature'][:50])

# Save combined feature importance - HARDCODED PATH
if feature_importance_results:
    all_importance = []
    for street, imp_df in feature_importance_results.items():
        imp_df = imp_df.copy()
        imp_df['street'] = street
        all_importance.append(imp_df)
    
    combined = pd.concat(all_importance, ignore_index=True)
    spark_importance = spark.createDataFrame(combined)
    spark_importance.write.mode('overwrite').parquet('/Volumes/pokerml/default/data/results/sp5_feature_importance')
    print(f"\n   Saved: /Volumes/pokerml/default/data/results/sp5_feature_importance")

In [ ]:
# Summary
elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("SP-5 Enhanced Profit Model COMPLETE!")
print("=" * 80)
print(f"\nRuntime: {elapsed/60:.1f} minutes")
print(f"Models trained: {len(best_models)} streets")

print("\n" + "-" * 80)
print("BEST MODELS:")
print("-" * 80)
for street, info in best_models.items():
    print(f"   {street}: {info['model_type']}")
    print(f"      RMSE={info['rmse']:.4f}, MAE={info['mae']:.4f}, R2={info['r2']:.4f}")
    print(f"      Explained Var={info['explained_variance']:.4f}, Max Error={info['max_error']:.4f}")

print("\n" + "-" * 80)
print("MODEL CONFIGURATION:")
print("-" * 80)
print("   Ridge: L2 regularization (alpha=1.0)")
print("   ElasticNet: L1+L2 mixed (alpha=0.5, l1_ratio=0.5)")
print("   RandomForest: 200 trees, max_depth=15, feature subsampling")
print("   GradientBoosting: 200 trees, max_depth=7, subsample=0.8")
print("   HistGradientBoosting: 200 iter, max_depth=10, L2=0.1, early_stopping")

print("\n" + "-" * 80)
print("OUTPUT - MLflow Model Registry (Unity Catalog):")
print("-" * 80)
for street in best_models.keys():
    print(f"   - {MODEL_REGISTRY_PREFIX}.sp5_profit_{street}")

print("\n" + "-" * 80)
print("OUTPUT - UC Volume:")
print("-" * 80)
for street in best_models.keys():
    print(f"   - {MODELS_DIR}sp5_{street}_metadata/")
print(f"   - {RESULTS_DIR}sp5_model_comparison/")
print(f"   - {RESULTS_DIR}sp5_feature_importance/")

print("\n" + "-" * 80)
print("METRICS LOGGED TO MLFLOW:")
print("-" * 80)
print("   Regression: rmse, mse, mae, r2, explained_variance, max_error, mape")
print("   Artifacts: Actual vs Predicted plots, Residual plots")

print("\n" + "-" * 80)
print("DATA LINEAGE:")
print("-" * 80)
print(f"   Input: SP-4 output ({SP4_INPUT_PATH})")
print(f"   Output: SP-5 profit models (MLflow Registry: {MODEL_REGISTRY_PREFIX})")

print("\n" + "-" * 80)
print("USAGE EXAMPLE:")
print("-" * 80)
print("   # Load model from MLflow (Unity Catalog)")
print("   import mlflow")
print(f"   model = mlflow.sklearn.load_model('models:/{MODEL_REGISTRY_PREFIX}.sp5_profit_preflop/latest')")
print("   ")
print("   # Predict profit")
print("   profit_pred = model.predict(X_new)")

print(f"\n[SUCCESS] Profit model training complete!")

<cell_type>markdown</cell_type>---
## Archived SparkML Version
The original SparkML implementation is archived in git history.
Converted to sklearn due to Databricks Serverless 100MB model size limit.